# Étude Approfondie : Données Socio-Économiques (INSEE)

## 1. Introduction et Objectifs
L'objectif de ce notebook est d'explorer le dataset des revenus de l'INSEE afin de sélectionner les indicateurs les plus pertinents pour expliquer l'adoption des véhicules électriques.

## 2. Chargement et Aperçu des données

In [ ]:
import pandas as pd
from src.loading import load_revenu_data

url_revenus='https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab'
df_revenus = load_revenu_data(url_revenus)

print(f"Le dataset contient {df_revenus.shape[0]} communes et {df_revenus.shape[1]} variables.")
df_revenus.sample(5)

Observation : Le dataset est très large (57 colonnes). Beaucoup de variables sont redondantes (revenu déclaré vs revenu disponible).

## 3. Sélection des variables d'intérêt

Avant de sélectionner nos indicateurs, il est essentiel de comprendre la nomenclature de l'INSEE.

In [ ]:
list(df_revenus.columns)

#### Signification des préfixes

- [DISP] : Revenu disponible des ménages
C'est le revenu après impôts directs et prestations sociales. Il correspond au pouvoir d’achat réel.
Note : Pour analyser l'adoption d'un bien coûteux comme le VE, ce préfixe est le plus pertinent.

- [DEC] : Revenu déclaré
C'est le revenu fiscal brut avant redistribution complète.

#### Description des informations contenues dans la base de données

La base de données regroupe plusieurs familles d’indicateurs permettant de caractériser les communes :

- **Identifiants géographiques** : permettent de localiser et d’identifier chaque commune de manière unique.
- **Taille et population fiscale** : décrivent la dimension démographique et fiscale des territoires.
- **Niveau de vie** : renseignent sur les revenus des habitants.
- **Dispersion et inégalités** : mesurent les écarts de revenus au sein de la population.
- **Origine des revenus** : détaillent la structure des sources de revenus (salaires, pensions, prestations, etc.).

Nous isolons les variables qui serviront de caractéristiques (features) dans notre modèle.

In [ ]:
# Sélection basée sur l'analyse de pertinence métier
var_selec = [
    'Code géographique',
    '[DISP] Médiane (€)',
    '[DISP] Rapport interdécile 9ᵉ décile/1ᵉʳ decile',
    '[DISP] Iice de Gini',
    '[DISP] Nbre de ménages fiscaux',
    '[DISP] Nbre de personnes dans les ménages fiscaux',
    '[DISP] Part des revenus d’activité (%)',
    '[DISP] dont part des iemnités de chômage (%)',
    "[DISP] Part de l'ensemble des prestations sociales (%)",
    '[DISP] dont part des revenus des activités non salariées (%)',
    '[DISP] Part des revenus du patrimoine et autres revenus (%)',
    '[DISP] Part des pensions, retraites et rentes (%)'
]

df_filtre = df_revenus[var_selec].copy()
df_filtre.sample(5)

## 4. Qualité des données

In [ ]:
# Vérification des types (attendus en float64 pour les calculs)
df_filtre.info()

## 5. Analyse de corrélation et affinage

Avant de valider notre sélection, nous étudions la corrélation de Spearman entre nos indicateurs. L'objectif est d'éliminer les variables trop corrélées entre elles pour éviter la redondance d'information dans nos modèles prédictifs.

In [ ]:
from src.utils import afficher_matrice_correlation, identifier_fortes_correlations

# Affichage de la heatmap
matrix = afficher_matrice_correlation(df_filtre)

# Identification des doublons statistiques
paires = identifier_fortes_correlations(matrix, threshold=0.70)
for p in paires:
    print(f"! Forte corrélation : {p['v1']} <--> {p['v2']} ({p['val']:.2f})")


In [ ]:
# Sélection finale validée pour le pipeline de préparation
var_finale_revenu = [
    'Code géographique',
    '[DISP] Médiane (€)',
    '[DISP] Iice de Gini',
    '[DISP] Nbre de ménages fiscaux',
    '[DISP] Nbre de personnes dans les ménages fiscaux',
    '[DISP] Part des revenus d’activité (%)',
    '[DISP] dont part des revenus des activités non salariées (%)',
    '[DISP] Part des revenus du patrimoine et autres revenus (%)'
]

df_revenu_final = df_filtre[var_finale_revenu]

L'analyse de corrélation a montré une colinéarité quasi parfaite ($r \approx 0.99$) entre le nombre de ménages et le nombre de personnes.

Nous décidons pour le moment de garder '[DISP] Nbre de ménages fiscaux' et '[DISP] Nbre de personnes dans les ménages fiscaux' car elles pourraient servir à normaliser de nouvelles variables (ex: indicateurs par habitant ou par foyer). Cependant, nous nous séparerons probablement de l'une des deux lors de la phase finale de modélisation pour éviter de biaiser les coefficients du modèle.

# brouillon

3. Indice de précarité
chomage + minima_sociaux + prestations_sociales
(normalisé)

à créer :
Ratio actifs / retraités
part_activite / part_retraites
👉 dynamisme démographique.